# Notebook 4.1: Exploring and Selecting Data with Pandas

**Companion to Chapter 4: Implementing Data Pre-processing in Python**  
*Machine Learning with Python: Principles and Practical Techniques*

> **Estimated time:** 35–45 minutes  
> **Level:** Beginner  
> **Environment:** Google Colab or Jupyter Notebook

---

## Related chapter ideas

This notebook supports the chapter discussion of Python libraries for machine learning, reading a dataset, inspecting its structure, and selecting rows and columns with Pandas.

## Learning objectives

By the end of this notebook, you will be able to:

1. load a tabular dataset into a Pandas `DataFrame`;
2. inspect its structure, columns, data types, and summary statistics;
3. distinguish label-based selection with `loc` from position-based selection with `iloc`;
4. filter rows using one or more conditions;
5. separate input features from a target variable; and
6. identify data-quality questions that should be addressed before modeling.


## What will you build?

You will explore a small **student success dataset** containing academic engagement, learning behavior, and course outcome information. By the end, you will create a model-ready view containing selected input features and a target variable.

The dataset is intentionally realistic: some values are missing, one record is duplicated, and one category is written inconsistently. In this notebook, we will **detect** these issues but not repair them. Data cleaning is the focus of Notebook 4.2.

> **Responsible practice:** The dataset is synthetic and contains no real student records. Real educational data must be handled according to institutional privacy policies and should not be used to make high-stakes decisions without appropriate human oversight.


## 1. Import the libraries


In [ ]:
from io import StringIO

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.precision", 2)

print("Pandas version:", pd.__version__)


### Why these libraries?

- **Pandas** provides the `DataFrame`, our main structure for tabular data.
- **NumPy** supports numerical operations and missing-value representations.
- **Matplotlib** helps us see patterns that may be difficult to notice in a table.
- **StringIO** lets this notebook load its embedded CSV data without requiring a separate download.


## 2. Load the dataset


In [ ]:
student_csv = """student_id,study_hours,attendance_pct,previous_score,learning_mode,programming_experience,assignments_submitted,final_score,passed
S001,5.5,92,78,In-person,Beginner,9,84,Yes
S002,3.0,75,65,Online,No prior experience,7,68,Yes
S003,1.5,61,58,Hybrid,No prior experience,5,55,No
S004,6.0,95,88,In-person,Intermediate,10,91,Yes
S005,2.0,70,62,Online,Beginner,6,63,Yes
S006,4.5,85,74,Hybrid,Beginner,8,79,Yes
S007,1.0,55,51,Online,No prior experience,4,48,No
S008,7.0,98,91,In-person,Advanced,10,95,Yes
S009,3.5,82,69,Hybrid,Beginner,8,73,Yes
S010,2.5,67,60,Online,No prior experience,6,59,No
S011,5.0,90,81,In-person,Intermediate,9,86,Yes
S012,4.0,88,76,Hybrid,Beginner,8,80,Yes
S013,2.0,,57,Online,No prior experience,5,54,No
S014,6.5,96,89,In-person,Advanced,10,93,Yes
S015,3.0,78,,Hybrid,Beginner,7,70,Yes
S016,1.5,63,55,Online,No prior experience,4,52,No
S017,5.5,91,83,In-person,Intermediate,9,88,Yes
S018,4.0,84,72,hybrid,Beginner,8,77,Yes
S019,2.5,72,64,Online,Beginner,6,65,Yes
S020,6.0,94,86,In-person,Advanced,10,90,Yes
S021,3.5,80,70,Hybrid,Beginner,7,72,Yes
S022,1.0,58,49,Online,No prior experience,3,45,No
S023,4.5,87,75,In-person,Intermediate,9,82,Yes
S024,2.0,69,59,Online,No prior experience,5,57,No
S025,5.0,89,80,Hybrid,Intermediate,9,85,Yes
S026,3.0,76,67,Online,Beginner,7,69,Yes
S027,6.5,97,90,In-person,Advanced,10,94,Yes
S028,1.5,60,53,Online,No prior experience,4,50,No
S029,4.0,83,73,Hybrid,Beginner,8,78,Yes
S030,2.5,74,63,Online,Beginner,6,64,Yes
S030,2.5,74,63,Online,Beginner,6,64,Yes
"""

students = pd.read_csv(StringIO(student_csv))
print("Dataset loaded successfully.")


> **In your own project:** You would usually load a file with `pd.read_csv("filename.csv")`. After loading data, always verify that the number of rows, columns, and column names are sensible.


## 3. Take a first look


In [ ]:
print("Shape (rows, columns):", students.shape)
display(students.head())


In [ ]:
print("Column names:")
print(students.columns.tolist())

print("\nData types:")
display(students.dtypes.to_frame("dtype"))


### Pause and predict

Before running the next cell, consider:

1. Which columns are numerical?
2. Which columns are categorical?
3. Which column could serve as a binary classification target?
4. Should `student_id` be treated as a predictive feature? Why or why not?


In [ ]:
students.info()


### Key insight

`info()` provides a compact structural audit. Compare the **non-null count** of each column with the total number of rows. A lower count signals missing data.


## 4. Summarize numerical and categorical columns


In [ ]:
display(students.describe().T)


In [ ]:
display(students.describe(include="object").T)


Summary statistics are useful, but they do not explain everything. For example, a valid-looking average can hide missing values, duplicate records, unusual values, or inconsistent category labels.


## 5. Select columns


In [ ]:
# Select one column as a Series.
attendance = students["attendance_pct"]
print(type(attendance))
display(attendance.head())


In [ ]:
# Select multiple columns as a DataFrame.
engagement_view = students[[
    "student_id",
    "study_hours",
    "attendance_pct",
    "assignments_submitted",
    "passed",
]]

print(type(engagement_view))
display(engagement_view.head())


### Series versus DataFrame

- `students["attendance_pct"]` returns a one-dimensional `Series`.
- `students[["attendance_pct"]]` returns a two-dimensional `DataFrame`.

This distinction matters because many machine-learning methods expect a two-dimensional feature matrix.


## 6. Select data by position with `iloc`


In [ ]:
# First row
display(students.iloc[0])

# First five rows and first four columns
display(students.iloc[0:5, 0:4])

# Selected, non-adjacent rows and columns
display(students.iloc[[0, 3, 7], [0, 1, 2, 7]])


`iloc` uses **integer positions**. Remember that Python starts counting at zero, and the stopping position in a slice is excluded.

For example, `students.iloc[0:5, 0:4]` returns rows 0–4 and columns 0–3.


## 7. Select data by labels with `loc`


In [ ]:
# Rows 0 through 4 and named columns.
# Unlike iloc slicing, both endpoints in this loc row slice are included.
display(students.loc[0:4, ["student_id", "study_hours", "passed"]])


### `loc` or `iloc`?

| Selector | Uses | Example |
|---|---|---|
| `loc` | row and column labels | `students.loc[0:4, ["study_hours", "passed"]]` |
| `iloc` | integer positions | `students.iloc[0:5, [1, 8]]` |

Use `loc` when column names make your intention clearer. Use `iloc` when position itself is meaningful.


## 8. Filter rows using conditions


In [ ]:
# Students who studied at least five hours per week.
high_study = students.loc[
    students["study_hours"] >= 5,
    ["student_id", "study_hours", "attendance_pct", "final_score"],
]
display(high_study)


In [ ]:
# Combine conditions with & (and).
# Place each condition inside parentheses.
engaged_students = students.loc[
    (students["attendance_pct"] >= 85)
    & (students["assignments_submitted"] >= 9),
    ["student_id", "attendance_pct", "assignments_submitted", "final_score"],
]
display(engaged_students)


In [ ]:
# Filter categorical values with isin().
flexible_modes = students.loc[
    students["learning_mode"].isin(["Online", "Hybrid"]),
    ["student_id", "learning_mode", "study_hours", "passed"],
]
display(flexible_modes.head(10))


> **Think like an ML practitioner:** Filtering should answer a clearly stated question. Repeatedly filtering until the data supports a preferred conclusion can introduce confirmation bias.


## 9. Explore simple patterns visually


In [ ]:
mode_counts = students["learning_mode"].value_counts(dropna=False)

ax = mode_counts.plot(
    kind="bar",
    color="#4C78A8",
    edgecolor="black",
    figsize=(7, 4),
)
ax.set_title("Students by Learning Mode")
ax.set_xlabel("Learning mode")
ax.set_ylabel("Number of records")
ax.tick_params(axis="x", rotation=0)
plt.tight_layout()
plt.show()

display(mode_counts.to_frame("count"))


### What did the visualization reveal?

Look closely at the category labels. `Hybrid` and `hybrid` appear separately even though they likely describe the same learning mode. This is a data-quality issue—not evidence of a new category.


In [ ]:
ax = students.plot.scatter(
    x="study_hours",
    y="final_score",
    c="attendance_pct",
    cmap="viridis",
    s=70,
    edgecolor="black",
    alpha=0.8,
    figsize=(7, 5),
)
ax.set_title("Study Hours and Final Score")
ax.set_xlabel("Weekly study hours")
ax.set_ylabel("Final score")
plt.tight_layout()
plt.show()


The scatter plot suggests an association, but it does not establish causation. Other factors may influence final scores, and repeated records can visually distort a pattern.


## 10. Perform a preliminary data-quality audit


In [ ]:
quality_report = pd.DataFrame({
    "dtype": students.dtypes.astype(str),
    "missing_count": students.isna().sum(),
    "missing_percent": students.isna().mean().mul(100).round(1),
    "unique_values": students.nunique(dropna=True),
})

display(quality_report)
print("Duplicate rows:", students.duplicated().sum())


In [ ]:
# Inspect unique values in categorical columns.
categorical_columns = students.select_dtypes(include="object").columns

for column in categorical_columns:
    print(f"{column}: {students[column].dropna().unique().tolist()}")


### Issues detected—but not yet repaired

You should have discovered:

- missing values in `attendance_pct` and `previous_score`;
- a duplicate record;
- inconsistent capitalization in `learning_mode`; and
- an identifier, `student_id`, that should not automatically become a predictive feature.

Notebook 4.2 will address these issues systematically.


## 11. Separate features and target


In [ ]:
# Select candidate input features.
feature_columns = [
    "study_hours",
    "attendance_pct",
    "previous_score",
    "learning_mode",
    "programming_experience",
    "assignments_submitted",
]

X = students.loc[:, feature_columns].copy()
y = students.loc[:, "passed"].copy()

print("Feature matrix X:", X.shape)
print("Target vector y:", y.shape)
display(X.head())
display(y.head().to_frame())


### Why were some columns excluded?

- `student_id` is an identifier, not a meaningful measure of learning.
- `final_score` directly determines or strongly overlaps with the outcome `passed`. Using it to predict `passed` would create **target leakage**.

Feature selection must be based on what information would genuinely be available at prediction time—not simply on which columns produce the highest accuracy.


## 12. Guided practice


Complete the following tasks before opening the solution.

1. Select `student_id`, `previous_score`, and `final_score` for the first eight rows.
2. Find students with attendance below 70% who did not pass.
3. Count records for each value of `programming_experience`.
4. Calculate the mean final score for each `learning_mode` value.


In [ ]:
# Write your solution here.


<details>
<summary><strong>Open the suggested solution</strong></summary>

```python
# 1. First eight rows and selected columns
display(students.loc[0:7, ["student_id", "previous_score", "final_score"]])

# 2. Low attendance and did not pass
display(students.loc[
    (students["attendance_pct"] < 70) & (students["passed"] == "No"),
    ["student_id", "attendance_pct", "passed"]
])

# 3. Records by experience level
display(students["programming_experience"].value_counts())

# 4. Mean final score by learning mode
display(students.groupby("learning_mode")["final_score"].mean())
```

</details>


## 13. Challenge: Build a focused analysis view


Create a DataFrame named `support_review` containing students who meet **either** of these conditions:

- attendance is below 75%, **or**
- fewer than six assignments were submitted.

Keep only `student_id`, `attendance_pct`, `assignments_submitted`, `learning_mode`, and `passed`. Sort the result by attendance from lowest to highest.

Then answer: **How could such a view support students without unfairly labeling or penalizing them?**


In [ ]:
# Write your challenge solution here.


<details>
<summary><strong>Open the suggested challenge solution</strong></summary>

```python
support_review = students.loc[
    (students["attendance_pct"] < 75)
    | (students["assignments_submitted"] < 6),
    [
        "student_id",
        "attendance_pct",
        "assignments_submitted",
        "learning_mode",
        "passed",
    ],
].sort_values("attendance_pct")

display(support_review)
```

The view could help educators offer optional resources or personal outreach. It should not be treated as proof that a student will fail, and human judgment should remain part of any intervention.

</details>


## 14. Reflection


Write brief answers in your own words.

1. What is the difference between `loc` and `iloc`?
2. Why should you inspect data types and missing-value counts immediately after loading data?
3. Why is `final_score` unsuitable for predicting `passed` in this example?
4. What problem could arise if `Hybrid` and `hybrid` remain separate categories?
5. What additional information would you need before using this dataset to support real students?


## 15. Key takeaways


- A Pandas `DataFrame` organizes tabular data into labeled rows and columns.
- `head()`, `info()`, `describe()`, and `value_counts()` provide complementary views of a dataset.
- `iloc` selects by integer position; `loc` selects by labels or conditions.
- Boolean conditions allow us to create focused views of the data.
- Visual inspection can reveal patterns and data-quality problems.
- Features and targets must be selected with attention to identifiers, availability at prediction time, and leakage.
- Data exploration should reveal questions—not hide uncertainty.

### Looking ahead

In **Notebook 4.2: Cleaning Real-World Data**, you will repair missing values, remove duplicate records, standardize inconsistent categories, examine outliers, and validate the cleaned dataset.
